In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())

options = Options()
# options.add_argument("--headless")  # Пока без headless для отладки
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:151.0) Gecko/20100101 Firefox/151.0")

print("🚀 Запускаем браузер...")
driver = webdriver.Chrome(service=service, options=options)

def scroll_until_end(driver, max_scrolls=50):
    """Прокручивает список до тех пор, пока не перестанут появляться новые элементы."""
    wait = WebDriverWait(driver, 10)
    
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul.search-list-view__list")))
    except:
        print("    ⚠️ Список не загрузился")
        return []
    
    time.sleep(3)
    
    seen_ids = set()
    all_data = []
    last_count = 0
    no_change_count = 0
    
    for i in range(max_scrolls):
        items = driver.find_elements(By.CSS_SELECTOR, "ul.search-list-view__list > li")
        
        new_found = 0
        for item in items:
            try:
                body = item.find_element(By.CSS_SELECTOR, "[data-coordinates]")
                item_id = body.get_attribute("data-id") or body.get_attribute("data-coordinates")
                
                if item_id and item_id not in seen_ids:
                    seen_ids.add(item_id)
                    coords_str = body.get_attribute("data-coordinates")
                    
                    # Название
                    try:
                        name = item.find_element(By.CSS_SELECTOR, "[class*='name'], [class*='title']").text.strip()
                    except:
                        name = "N/A"
                    
                    # Адрес
                    try:
                        address = item.find_element(By.CSS_SELECTOR, "[class*='address'], [class*='subtitle']").text.strip()
                    except:
                        address = ""
                    
                    # Категория
                    try:
                        category = item.find_element(By.CSS_SELECTOR, "[class*='category']").text.strip()
                    except:
                        category = ""
                    
                    lon_str, lat_str = coords_str.split(',')
                    
                    all_data.append({
                        'name': name,
                        'address': address,
                        'category': category,
                        'lat': float(lat_str),
                        'lon': float(lon_str),
                    })
                    new_found += 1
            except:
                continue
        
        print(f"    Прокрутка {i+1}: +{new_found} новых, всего {len(all_data)}")
        
        if new_found == 0:
            no_change_count += 1
            if no_change_count >= 3:
                print("    Достигнут конец списка")
                break
        else:
            no_change_count = 0
        
        # Скроллим к последнему элементу
        if items:
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", items[-1])
            time.sleep(2)
    
    return all_data

def collect_square(query, lat, lon, zoom=14):
    """Собирает ВСЕ точки в квадрате."""
    encoded = query.replace(' ', '%20')
    url = f"https://yandex.by/maps/157/minsk/search/{encoded}/?ll={lon}%2C{lat}&z={zoom}"
    print(f"    URL: {url}")
    driver.get(url)
    return scroll_until_end(driver)

# ============================================================
# УЛУЧШЕННАЯ СЕТКА: 4x4 = 16 квадратов
# ============================================================
lats = [53.84, 53.88, 53.92, 53.96]
lons = [27.44, 27.52, 27.60, 27.68]
queries = ["банкомат", "отделение банка"]

all_data = []
total = len(lats) * len(lons) * len(queries)

print(f"\n🔍 СБОР ДАННЫХ: сетка {len(lats)}x{len(lons)} = {len(lats)*len(lons)} квадратов")
print(f"   Всего запросов: {total}")
print("="*60)

current = 0
for lat in lats:
    for lon in lons:
        for query in queries:
            current += 1
            print(f"\n[{current}/{total}] {query} в ({lat}, {lon})")
            
            try:
                results = collect_square(query, lat, lon)
                
                if results:
                    for r in results:
                        r['query'] = query
                    all_data.extend(results)
                    print(f"    ✅ +{len(results)}")
                else:
                    print(f"    ❌ Пусто")
            except Exception as e:
                print(f"    ⚠️ Ошибка: {e}")
            
            time.sleep(2)

driver.quit()

# Сохраняем
if all_data:
    df = pd.DataFrame(all_data)
    df = df.drop_duplicates(subset=['name', 'address', 'lat', 'lon'])
    df.to_csv('minsk_banks_improved.csv', index=False)
    print(f"\n📊 ИТОГО: {len(df)} уникальных точек")
else:
    print("\n❌ Ничего не собрано.")

🚀 Запускаем браузер...

🔍 СБОР ДАННЫХ: сетка 4x4 = 16 квадратов
   Всего запросов: 32

[1/32] банкомат в (53.84, 27.44)
    URL: https://yandex.by/maps/157/minsk/search/банкомат/?ll=27.44%2C53.84&z=14
    Прокрутка 1: +9 новых, всего 9
    Прокрутка 2: +4 новых, всего 13
    Прокрутка 3: +4 новых, всего 17
    Прокрутка 4: +4 новых, всего 21
    Прокрутка 5: +4 новых, всего 25
    Прокрутка 6: +1 новых, всего 26
    Прокрутка 7: +4 новых, всего 30
    Прокрутка 8: +4 новых, всего 34
    Прокрутка 9: +4 новых, всего 38
    Прокрутка 10: +4 новых, всего 42
    Прокрутка 11: +4 новых, всего 46
    Прокрутка 12: +4 новых, всего 50
    Прокрутка 13: +1 новых, всего 51
    Прокрутка 14: +4 новых, всего 55
    Прокрутка 15: +4 новых, всего 59
    Прокрутка 16: +4 новых, всего 63
    Прокрутка 17: +4 новых, всего 67
    Прокрутка 18: +4 новых, всего 71
    Прокрутка 19: +4 новых, всего 75
    Прокрутка 20: +1 новых, всего 76
    Прокрутка 21: +4 новых, всего 80
    Прокрутка 22: +4 новых, всег

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Загружаем улучшенный датасет
df = pd.read_csv('minsk_banks_improved.csv')
print(f"Загружено: {len(df)} точек")

# Извлекаем название банка
def extract_bank(name):
    if pd.isna(name):
        return 'Неизвестно'
    name = str(name).strip()
    
    banks_map = {
        'беларусбанк': 'Беларусбанк',
        'асб беларусбанк': 'Беларусбанк',
        'белагропромбанк': 'Белагропромбанк',
        'приорбанк': 'Приорбанк',
        'альфа-банк': 'Альфа-Банк',
        'альфа банк': 'Альфа-Банк',
        'альфа private': 'Альфа-Банк',
        'альфабанк, банкомат': 'Альфа-Банк',
        'белинвестбанк': 'Белинвестбанк',
        'мтбанк': 'МТБанк',
        'сбер банк': 'Сбер Банк',
        'сбер': 'Сбер Банк',
        'сбербанк': 'Сбер Банк',
        'сберБанк': 'Сбер Банк',
        'сбербизнес': 'Сбер Банк',
        'бпс-сбербанк': 'Сбер Банк',
        'сбер Первый': 'Сбер Банк',
        'бнб-банк': 'БНБ-Банк',
        'бнб': 'БНБ-Банк',
        'белвэб': 'БелВЭБ',
        'банк белвэб': 'БелВЭБ',
        'технобанк': 'Технобанк',
        'банк дабрабыт': 'Банк Дабрабыт',
        'белгазпромбанк': 'Белгазпромбанк',
        'банк втб': 'Банк ВТБ',
        'втб': 'Банк ВТБ',
        'бсб банк': 'БСБ Банк',
        'статус банк': 'Статус Банк',
        'атм белинвестбанк': 'Белинвестбанк',
        'франсабанк': 'Франсабанк',
        'банк Решение,': 'Банк Решение',
        'решение': 'Банк Решение',
        'абсолютбанк': 'Абсолютбанк',
        'цептер банк': 'Цептер Банк',
        'zepter bank': 'Цептер Банк',
        'zepter Bank': 'Цептер Банк',
        'bsb bank': 'БСБ Банк',
        'neo bank': 'Neo Bank',
        'идея банк': 'Идея Банк',
        'белорусский народный': 'БНБ-Банк',
        'банк торговый': 'Банк Торговый капитал',
        'ррб-банк': 'РРБ-Банк',
        'ррб': 'РРБ-Банк',
        'паритетбанк': 'Паритетбанк',
    }
    
    name_lower = name.lower()
    for key, value in banks_map.items():
        if key in name_lower:
            return value
    
    # Если не нашли — первые 2 слова
    words = name.split()
    return ' '.join(words[:2]) if len(words) >= 2 else name

df['bank_name'] = df['name'].apply(extract_bank)

# Определяем тип точки
def determine_type(row):
    combined = f"{row.get('category', '')} {row['query']}".lower()
    if 'банкомат' in combined:
        return 'atm'
    elif 'отделение' in combined or 'банк' in combined:
        return 'office'
    return 'unknown'

df['type'] = df.apply(determine_type, axis=1)

# Создаём геодатафрейм
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
gdf_banks = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Score для взвешенного анализа
score_map = {'office': 3, 'atm': 2, 'unknown': 1}
gdf_banks['score'] = gdf_banks['type'].map(score_map).fillna(1)

# Статистика
print(f"\n📊 Статистика:")
print(f"   Всего: {len(gdf_banks)}")
print(f"   Банкоматов: {len(gdf_banks[gdf_banks['type'] == 'atm'])}")
print(f"   Отделений: {len(gdf_banks[gdf_banks['type'] == 'office'])}")

print(f"\n🏦 Топ-10 банков:")
for bank, count in gdf_banks['bank_name'].value_counts().head(10).items():
    atms = len(gdf_banks[(gdf_banks['bank_name'] == bank) & (gdf_banks['type'] == 'atm')])
    offices = len(gdf_banks[(gdf_banks['bank_name'] == bank) & (gdf_banks['type'] == 'office')])
    print(f"   {bank}: {count} (ATM: {atms}, Office: {offices})")

# Сохраняем
gdf_banks.to_file('minsk_banks_clean.geojson', driver='GeoJSON')
print(f"\n💾 Сохранено: minsk_banks_clean.geojson")

Загружено: 1661 точек

📊 Статистика:
   Всего: 1661
   Банкоматов: 1170
   Отделений: 491

🏦 Топ-10 банков:
   Беларусбанк: 419 (ATM: 260, Office: 159)
   Белинвестбанк: 162 (ATM: 145, Office: 17)
   Белагропромбанк: 162 (ATM: 123, Office: 39)
   Альфа-Банк: 137 (ATM: 119, Office: 18)
   Сбер Банк: 128 (ATM: 103, Office: 25)
   Приорбанк: 117 (ATM: 97, Office: 20)
   МТБанк: 84 (ATM: 43, Office: 41)
   БСБ Банк: 76 (ATM: 46, Office: 30)
   БелВЭБ: 74 (ATM: 58, Office: 16)
   Белгазпромбанк: 70 (ATM: 50, Office: 20)

💾 Сохранено: minsk_banks_clean.geojson


In [5]:
import osmnx as ox
import geopandas as gpd

print("🏢 Выгружаем жилые здания Минска из OpenStreetMap...")
print("   Это может занять 3-5 минут...")

# Загружаем все здания Минска
buildings = ox.features_from_place('Минск, Беларусь', tags={'building': True})

# Оставляем только жилые
residential = buildings[buildings['building'].isin([
    'apartments', 'residential', 'house', 'detached', 
    'terrace', 'semidetached_house', 'yes'
])].copy()

print(f"   Загружено зданий: {len(buildings)}")
print(f"   Из них жилых: {len(residential)}")

# Оставляем только полигоны (не точки)
residential = residential[residential.geometry.type.isin(['Polygon', 'MultiPolygon'])]
print(f"   Жилых полигонов: {len(residential)}")

# Сохраняем
residential.to_file('minsk_residential_buildings.geojson', driver='GeoJSON')
print(f"💾 Сохранено: minsk_residential_buildings.geojson")

🏢 Выгружаем жилые здания Минска из OpenStreetMap...
   Это может занять 3-5 минут...
   Загружено зданий: 55094
   Из них жилых: 43892
   Жилых полигонов: 43688
💾 Сохранено: minsk_residential_buildings.geojson


In [6]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
import numpy as np

# Загружаем данные
gdf_houses = gpd.read_file('minsk_residential_buildings.geojson')
gdf_banks = gpd.read_file('minsk_banks_clean.geojson')

print(f"Исходно домов: {len(gdf_houses)}")
print(f"Банковских точек: {len(gdf_banks)}")

# Границы нашего поиска банков (те же, что в сетке Selenium)
LAT_MIN, LAT_MAX = 53.82, 53.98
LON_MIN, LON_MAX = 27.38, 27.70

# Фильтруем дома по bounding box
gdf_houses_filtered = gdf_houses.cx[LON_MIN:LON_MAX, LAT_MIN:LAT_MAX].copy()

print(f"\nПосле фильтрации домов: {len(gdf_houses_filtered)}")
print(f"   Удалено: {len(gdf_houses) - len(gdf_houses_filtered)} домов за пределами области поиска")

# --- ПЕРЕСЧИТЫВАЕМ ДОСТУПНОСТЬ ---
gdf_banks_m = gdf_banks.to_crs('EPSG:32635')
gdf_houses_m = gdf_houses_filtered.to_crs('EPSG:32635')

# Все банки в мультипойнт
all_points = MultiPoint(gdf_banks_m.geometry.tolist())

print("📏 Пересчитываем расстояния...")
centroids = gdf_houses_m.geometry.centroid
distances = centroids.apply(lambda c: c.distance(all_points))

gdf_houses_m['distance_to_nearest'] = distances
gdf_houses_filtered = gdf_houses_m.to_crs('EPSG:4326')

# Категории
def categorize_distance(d):
    if d <= 500:
        return 'Отлично (до 500м)'
    elif d <= 1000:
        return 'Хорошо (500м-1км)'
    elif d <= 2000:
        return 'Удовлетворительно (1-2км)'
    else:
        return 'Плохо (более 2км)'

gdf_houses_filtered['accessibility'] = gdf_houses_filtered['distance_to_nearest'].apply(categorize_distance)

# Статистика
print(f"\n📊 Доступность банковских услуг (только Минск):")
print(f"   Всего домов: {len(gdf_houses_filtered)}")
for cat in ['Отлично (до 500м)', 'Хорошо (500м-1км)', 'Удовлетворительно (1-2км)', 'Плохо (более 2км)']:
    count = len(gdf_houses_filtered[gdf_houses_filtered['accessibility'] == cat])
    pct = count / len(gdf_houses_filtered) * 100
    print(f"   {cat}: {count} ({pct:.1f}%)")

bad_pct = (gdf_houses_filtered['accessibility'] == 'Плохо (более 2км)').mean() * 100
print(f"\n🚨 Домов с плохой доступностью: {bad_pct:.1f}%")

# Сохраняем исправленную версию
gdf_houses_filtered.to_file('minsk_accessibility.geojson', driver='GeoJSON')
print(f"\n💾 Сохранено: minsk_accessibility.geojson")

Исходно домов: 43688
Банковских точек: 1654

После фильтрации домов: 39398
   Удалено: 4290 домов за пределами области поиска
📏 Пересчитываем расстояния...

📊 Доступность банковских услуг (только Минск):
   Всего домов: 39398
   Отлично (до 500м): 26795 (68.0%)
   Хорошо (500м-1км): 10998 (27.9%)
   Удовлетворительно (1-2км): 1601 (4.1%)
   Плохо (более 2км): 4 (0.0%)

🚨 Домов с плохой доступностью: 0.0%

💾 Сохранено: minsk_accessibility.geojson


In [7]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import MultiPoint

# Загружаем данные
gdf_banks = gpd.read_file('minsk_banks_clean.geojson')
gdf_houses = gpd.read_file('minsk_accessibility.geojson')

print(f"Загружено: {len(gdf_houses)} домов, {len(gdf_banks)} банковских точек")

# Переименовываем колонки из accessibility → dist_all / cat_all
gdf_houses = gdf_houses.rename(columns={
    'distance_to_nearest': 'dist_all',
    'accessibility': 'cat_all',
})

# Переводим в метровую проекцию
gdf_banks_m = gdf_banks.to_crs('EPSG:32635')
gdf_houses_m = gdf_houses.to_crs('EPSG:32635')

# Разделяем банки по типам
banks_office = gdf_banks_m[gdf_banks_m['type'] == 'office']
banks_atm = gdf_banks_m[gdf_banks_m['type'] == 'atm']

print(f"   Отделений: {len(banks_office)}")
print(f"   Банкоматов: {len(banks_atm)}")

# Мультипойнты для быстрого расчёта
all_points = MultiPoint(gdf_banks_m.geometry.tolist())
office_points = MultiPoint(banks_office.geometry.tolist()) if len(banks_office) > 0 else None
atm_points = MultiPoint(banks_atm.geometry.tolist()) if len(banks_atm) > 0 else None

# Предрассчитываем расстояния
print("\n📏 Расчёт расстояний...")
centroids = gdf_houses_m.geometry.centroid
dist_office = centroids.apply(lambda c: c.distance(office_points)) if office_points else np.full(len(centroids), np.nan)
dist_atm = centroids.apply(lambda c: c.distance(atm_points)) if atm_points else np.full(len(centroids), np.nan)

# Добавляем расстояния в GeoDataFrame
gdf_houses['dist_office'] = dist_office
gdf_houses['dist_atm'] = dist_atm

# Функция категоризации
def categorize(d):
    if pd.isna(d):
        return 'Нет данных'
    if d <= 500:
        return 'Отлично (до 500м)'
    elif d <= 1000:
        return 'Хорошо (500м-1км)'
    elif d <= 2000:
        return 'Удовлетворительно (1-2км)'
    else:
        return 'Плохо (более 2км)'

gdf_houses['cat_all'] = gdf_houses['dist_all'].apply(categorize)
gdf_houses['cat_office'] = gdf_houses['dist_office'].apply(categorize)
gdf_houses['cat_atm'] = gdf_houses['dist_atm'].apply(categorize)

# Добавляем ближайший банк для каждого типа
def nearest_bank_name(centroid, banks_gdf):
    if len(banks_gdf) == 0:
        return ''
    distances = banks_gdf.geometry.distance(centroid)
    idx = distances.idxmin()
    return banks_gdf.loc[idx, 'bank_name']

# Для каждого дома найдём ближайший банк (офис и банкомат)
centroids = gdf_houses_m.geometry.centroid

print("🏦 Определяем ближайшие банки...")
if len(banks_office) > 0:
    nearest_office_idxs = centroids.apply(lambda c: banks_office.geometry.distance(c).idxmin())
    nearest_office = banks_office.loc[nearest_office_idxs, 'bank_name'].tolist()
else:
    nearest_office = [''] * len(centroids)

if len(banks_atm) > 0:
    nearest_atm_idxs = centroids.apply(lambda c: banks_atm.geometry.distance(c).idxmin())
    nearest_atm = banks_atm.loc[nearest_atm_idxs, 'bank_name'].tolist()
else:
    nearest_atm = [''] * len(centroids)

gdf_houses['nearest_office_bank'] = nearest_office
gdf_houses['nearest_atm_bank'] = nearest_atm

# Сохраняем результат
gdf_houses.to_file('minsk_houses_enriched.geojson', driver='GeoJSON')
print(f"\n💾 Сохранено: minsk_houses_enriched.geojson")
print(f"   Колонки: {gdf_houses.columns.tolist()}")
print(f"\n📊 Статистика:")
print(f"   Всего домов: {len(gdf_houses)}")
print(f"\n   Доступность (все точки):")
for cat in ['Отлично (до 500м)', 'Хорошо (500м-1км)', 'Удовлетворительно (1-2км)', 'Плохо (более 2км)']:
    count = len(gdf_houses[gdf_houses['cat_all'] == cat])
    pct = count / len(gdf_houses) * 100
    print(f"      {cat}: {count} ({pct:.1f}%)")

Загружено: 39398 домов, 1654 банковских точек
   Отделений: 486
   Банкоматов: 1168

📏 Расчёт расстояний...
🏦 Определяем ближайшие банки...

💾 Сохранено: minsk_houses_enriched.geojson
   Колонки: ['element', 'id', 'building', 'name', 'name:be', 'name:ru', 'leisure', 'name:uk', 'sport', 'addr:housenumber', 'addr:postcode', 'addr:street', 'entrance', 'addr:city', 'addr:city:ru', 'addr:country', 'brand', 'brand:wikidata', 'shop', 'capacity', 'opening_hours', 'operator', 'ref', 'name:zh', 'bunker_type', 'military', 'addr:housename', 'description', 'email', 'level', 'name:en', 'office', 'phone', 'start_date', 'website', 'building:levels', 'tickets:public_transport', 'amenity', 'man_made', 'access', 'payment:cash', 'payment:coins', 'payment:electronic_purses', 'payment:mastercard', 'payment:visa', 'official_short_type', 'power', 'substation', 'voltage', 'location', 'building:material', 'contact:email', 'contact:phone', 'contact:website', 'check_date', 'contact:facebook', 'contact:instagram',

In [8]:
import json
import pandas as pd
import geopandas as gpd

# Загружаем обогащённые данные
gdf_houses = gpd.read_file('minsk_houses_enriched.geojson')
gdf_banks = gpd.read_file('minsk_banks_clean.geojson')

# Подготавливаем данные для JavaScript
# Дома: координаты центроидов + категории + расстояния
gdf_houses_m = gdf_houses.to_crs('EPSG:32635')
centroids_m = gdf_houses_m.geometry.centroid
centroids = gpd.GeoSeries(centroids_m, crs='EPSG:32635').to_crs('EPSG:4326')

houses_data = []
for idx, row in gdf_houses.iterrows():
    houses_data.append({
        'lat': centroids.iloc[idx].y,
        'lon': centroids.iloc[idx].x,
        'dist_all': round(row['dist_all'], 0) if not pd.isna(row['dist_all']) else None,
        'dist_office': round(row['dist_office'], 0) if not pd.isna(row['dist_office']) else None,
        'dist_atm': round(row['dist_atm'], 0) if not pd.isna(row['dist_atm']) else None,
        'cat_all': row['cat_all'],
        'cat_office': row['cat_office'],
        'cat_atm': row['cat_atm'],
        'nearest_office': row['nearest_office_bank'],
        'nearest_atm': row['nearest_atm_bank'],
    })

# Банки: координаты + информация
banks_data = []
for idx, row in gdf_banks.iterrows():
    banks_data.append({
        'lat': row['lat'],
        'lon': row['lon'],
        'name': row['name'],
        'bank_name': row['bank_name'],
        'type': row['type'],
        'address': row.get('address', ''),
    })

# Список банков для фильтра
bank_list = sorted(gdf_banks['bank_name'].dropna().unique().tolist())

# Статистика для начального отображения
stats = {}
for mode, col in [('all', 'cat_all'), ('office', 'cat_office'), ('atm', 'cat_atm')]:
    counts = gdf_houses[col].value_counts().to_dict()
    stats[mode] = {
        'total': len(gdf_houses),
        'excellent': counts.get('Отлично (до 500м)', 0),
        'good': counts.get('Хорошо (500м-1км)', 0),
        'ok': counts.get('Удовлетворительно (1-2км)', 0),
        'bad': counts.get('Плохо (более 2км)', 0),
    }

# Собираем всё в один JSON для вставки в HTML
dashboard_data = {
    'houses': houses_data,
    'banks': banks_data,
    'bankList': bank_list,
    'stats': stats,
}

# Сохраняем JSON
with open('dashboard_data.json', 'w', encoding='utf-8') as f:
    json.dump(dashboard_data, f, ensure_ascii=False)

print(f"💾 Сохранено: dashboard_data.json")
print(f"   Домов: {len(houses_data)}")
print(f"   Банков: {len(banks_data)}")
print(f"   Банков в списке: {len(bank_list)}")

💾 Сохранено: dashboard_data.json
   Домов: 39398
   Банков: 1654
   Банков в списке: 23


In [8]:
with open('dashboard_data.json', 'r', encoding='utf-8') as f:
    dashboard_data = json.load(f)

houses_json = json.dumps(dashboard_data['houses'], ensure_ascii=False)
banks_json = json.dumps(dashboard_data['banks'], ensure_ascii=False)
bank_list_json = json.dumps(dashboard_data['bankList'], ensure_ascii=False)
stats_json = json.dumps(dashboard_data['stats'], ensure_ascii=False)

html_template = '''<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Банковская доступность — Минск</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
    * { margin: 0; padding: 0; box-sizing: border-box; }
    body { font-family: 'Segoe UI', Arial, sans-serif; background: #1a1a2e; color: #eee; }
    
    #container { display: flex; height: 100vh; }
    
    #sidebar {
        width: 370px; min-width: 370px;
        background: #16213e; padding: 20px;
        display: flex; flex-direction: column; gap: 15px;
        overflow-y: auto; border-right: 2px solid #0f3460;
    }
    
    #map { flex: 1; height: 100vh; }
    
    .panel {
        background: #0f3460; border-radius: 10px; padding: 15px;
    }
    
    .panel h3 {
        margin-bottom: 10px; color: #e94560; font-size: 16px;
        border-bottom: 1px solid #533483; padding-bottom: 5px;
    }
    
    button {
        width: 100%; padding: 8px; margin: 5px 0;
        background: #e94560; color: #eee; border: none;
        border-radius: 5px; cursor: pointer; font-size: 14px;
        font-weight: bold; transition: background 0.3s;
    }
    
    button:hover { background: #c23152; }
    
    .stats-row {
        display: flex; justify-content: space-between;
        font-size: 14px; margin: 3px 0;
    }
    
    .stats-dot {
        display: inline-block; width: 10px; height: 10px;
        border-radius: 50%; margin-right: 5px;
    }
    
    .legend-item {
        display: flex; align-items: center; margin: 5px 0; font-size: 13px;
    }
    
    .legend-color {
        width: 20px; height: 20px; border-radius: 4px; margin-right: 8px;
    }
    
    .switch-container {
        display: flex; gap: 5px; flex-wrap: wrap;
    }
    
    .switch-btn {
        flex: 1; padding: 8px; text-align: center;
        background: #1a1a2e; border: 1px solid #533483;
        border-radius: 5px; cursor: pointer; font-size: 12px;
        transition: all 0.3s;
    }
    
    .switch-btn.active {
        background: #e94560; border-color: #e94560;
    }
    
    .summary-box {
        text-align: center; padding: 10px;
        background: #1a1a2e; border-radius: 5px;
    }
    
    .summary-box .big-number {
        font-size: 36px; font-weight: bold; color: #e94560;
    }
    
    .checkbox-list {
        max-height: 180px; overflow-y: auto;
        background: #1a1a2e; border-radius: 5px; padding: 8px;
    }
    
    .checkbox-item {
        display: flex; align-items: center; padding: 4px 0;
        font-size: 13px; cursor: pointer;
    }
    
    .checkbox-item input {
        margin-right: 8px; cursor: pointer;
        accent-color: #e94560;
    }
    
    .bank-count {
        margin-left: auto; color: #999; font-size: 11px;
    }
    
    .select-all-row {
        display: flex; gap: 10px; margin-bottom: 5px;
    }
    
    .select-all-row button {
        flex: 1; padding: 4px; font-size: 11px; background: #533483;
    }
</style>
</head>
<body>

<div id="container">
    <div id="sidebar">
        <h2>Банковская доступность</h2>
        <p style="font-size:12px; color:#999;">Минск, 1661 точка обслуживания</p>
        
        <div class="panel">
            <h3>Режим отображения</h3>
            <div class="switch-container" id="modeSwitch">
                <div class="switch-btn active" data-mode="all">Все точки</div>
                <div class="switch-btn" data-mode="office">Отделения</div>
                <div class="switch-btn" data-mode="atm">Банкоматы</div>
            </div>
        </div>
        
        <div class="panel">
            <h3>Фильтр по банкам</h3>
            <div class="select-all-row">
                <button onclick="selectAllBanks()">Выбрать все</button>
                <button onclick="deselectAllBanks()">Снять все</button>
            </div>
            <div class="checkbox-list" id="bankCheckboxes"></div>
            <button onclick="applyBankFilter()">Применить фильтр</button>
        </div>
        
        <div class="panel">
            <h3>Статистика</h3>
            <div id="statsContent"></div>
        </div>
        
        <div class="panel">
            <h3>Легенда</h3>
            <div class="legend-item">
                <div class="legend-color" style="background:#2ecc71;"></div>
                Отлично (до 500м)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#f1c40f;"></div>
                Хорошо (500м–1км)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#e67e22;"></div>
                Удовлетворительно (1–2км)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#e74c3c;"></div>
                Плохо (более 2км)
            </div>
            <hr style="border-color:#533483; margin:10px 0;">
            <div class="legend-item">
                <div class="legend-color" style="background:darkgreen;"></div>
                Отделение
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:darkblue;"></div>
                Банкомат
            </div>
        </div>
    </div>
    
    <div id="map"></div>
</div>

<script>
var housesData = __HOUSES__;
var banksData = __BANKS__;
var bankList = __BANKLIST__;
var statsData = __STATS__;

var currentMode = 'all';
var filteredBanks = ['all'];

// Количество точек для каждого банка
var bankCounts = {};
for (var i = 0; i < banksData.length; i++) {
    var bn = banksData[i].bank_name;
    bankCounts[bn] = (bankCounts[bn] || 0) + 1;
}

// Сортируем банки по количеству точек
bankList.sort(function(a, b) {
    return (bankCounts[b] || 0) - (bankCounts[a] || 0);
});

// Создаём чекбоксы
var checkboxContainer = document.getElementById('bankCheckboxes');
for (var i = 0; i < bankList.length; i++) {
    var bank = bankList[i];
    var count = bankCounts[bank] || 0;
    
    var label = document.createElement('label');
    label.className = 'checkbox-item';
    
    var checkbox = document.createElement('input');
    checkbox.type = 'checkbox';
    checkbox.value = bank;
    checkbox.checked = true;
    checkbox.className = 'bank-checkbox';
    
    label.appendChild(checkbox);
    label.appendChild(document.createTextNode(bank));
    
    var span = document.createElement('span');
    span.className = 'bank-count';
    span.textContent = count;
    label.appendChild(span);
    
    checkboxContainer.appendChild(label);
}

function selectAllBanks() {
    var checks = document.querySelectorAll('.bank-checkbox');
    for (var i = 0; i < checks.length; i++) checks[i].checked = true;
}

function deselectAllBanks() {
    var checks = document.querySelectorAll('.bank-checkbox');
    for (var i = 0; i < checks.length; i++) checks[i].checked = false;
}

// Карта
var map = L.map('map', {
    center: [53.9, 27.56],
    zoom: 12,
    zoomControl: true,
});

L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', {
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>',
    maxZoom: 18,
}).addTo(map);

var housesLayer = L.featureGroup().addTo(map);
var banksLayer = L.featureGroup().addTo(map);

var categoryColors = {
    'Отлично (до 500м)': '#2ecc71',
    'Хорошо (500м-1км)': '#f1c40f',
    'Удовлетворительно (1-2км)': '#e67e22',
    'Плохо (более 2км)': '#e74c3c',
    'Нет данных': '#95a5a6',
};

var HOUSE_SAMPLE = 5; // Каждый 5-й дом для производительности

function getDistField() { return 'dist_' + currentMode; }
function getCatField() { return 'cat_' + currentMode; }

function renderHouses() {
    housesLayer.clearLayers();
    
    var distField = getDistField();
    var catField = getCatField();
    
    for (var i = 0; i < housesData.length; i += HOUSE_SAMPLE) {
        var house = housesData[i];
        var dist = house[distField];
        var cat = house[catField];
        
        if (dist === null || dist === undefined) continue;
        
        var color = categoryColors[cat] || '#95a5a6';
        var marker = L.circleMarker([house.lat, house.lon], {
            radius: 4,
            fillColor: color,
            color: color,
            weight: 0.5,
            fillOpacity: 0.6,
        });
        
        marker.bindPopup(
            '<b>' + cat + '</b><br>' +
            'Расстояние: ' + Math.round(dist) + 'м<br>' +
            '<small>Ближ. офис: ' + (house.nearest_office || '—') + '</small><br>' +
            '<small>Ближ. банкомат: ' + (house.nearest_atm || '—') + '</small>'
        );
        
        housesLayer.addLayer(marker);
    }
}

function renderBanks() {
    banksLayer.clearLayers();
    
    // Если фильтр пустой — не показываем ничего
    if (filteredBanks.length === 0) return;
    
    for (var i = 0; i < banksData.length; i++) {
        var bank = banksData[i];
        
        if (filteredBanks.indexOf('all') === -1 && 
            filteredBanks.indexOf(bank.bank_name) === -1) continue;
        
        if (currentMode === 'office' && bank.type !== 'office') continue;
        if (currentMode === 'atm' && bank.type !== 'atm') continue;
        
        var color = bank.type === 'office' ? 'darkgreen' : 'darkblue';
        var marker = L.circleMarker([bank.lat, bank.lon], {
            radius: 6,
            fillColor: color,
            color: '#fff',
            weight: 1.5,
            fillOpacity: 0.95,
        });
        
        marker.bindPopup(
            '<b>' + bank.bank_name + '</b><br>' +
            (bank.type === 'office' ? 'Отделение' : 'Банкомат') + '<br>' +
            (bank.address || '')
        );
        
        banksLayer.addLayer(marker);
    }
}

function renderAll() {
    renderHouses();
    renderBanks();
    updateStats();
}

function updateStats() {
    var modeStats = statsData[currentMode];
    var total = modeStats.total;
    
    document.getElementById('statsContent').innerHTML =
        '<div class="summary-box">' +
        '<div class="big-number">' + total.toLocaleString() + '</div>' +
        '<div>домов проанализировано</div>' +
        '</div>' +
        '<div style="margin-top:10px;">' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#2ecc71;"></span>Отлично</span>' +
        '<span>' + modeStats.excellent.toLocaleString() + ' (' + (modeStats.excellent/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#f1c40f;"></span>Хорошо</span>' +
        '<span>' + modeStats.good.toLocaleString() + ' (' + (modeStats.good/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#e67e22;"></span>Удовл.</span>' +
        '<span>' + modeStats.ok.toLocaleString() + ' (' + (modeStats.ok/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#e74c3c;"></span>Плохо</span>' +
        '<span>' + modeStats.bad.toLocaleString() + ' (' + (modeStats.bad/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '</div>';
}

// Переключатели режима
document.querySelectorAll('#modeSwitch .switch-btn').forEach(function(btn) {
    btn.addEventListener('click', function() {
        document.querySelectorAll('#modeSwitch .switch-btn').forEach(function(b) { b.classList.remove('active'); });
        this.classList.add('active');
        currentMode = this.dataset.mode;
        renderAll();
    });
});

// Фильтр банков — пустой фильтр = пустая карта
function applyBankFilter() {
    var checks = document.querySelectorAll('.bank-checkbox');
    var selected = [];
    
    for (var i = 0; i < checks.length; i++) {
        if (checks[i].checked) {
            selected.push(checks[i].value);
        }
    }
    
    if (selected.length === 0) {
        filteredBanks = []; // Пустой фильтр — ничего не показываем
    } else if (selected.length === bankList.length) {
        filteredBanks = ['all']; // Все выбраны — показываем все
    } else {
        filteredBanks = selected; // Выборочный фильтр
    }
    
    renderAll();
}

renderAll();
</script>
</body>
</html>'''

html = html_template.replace('__HOUSES__', houses_json)
html = html.replace('__BANKS__', banks_json)
html = html.replace('__BANKLIST__', bank_list_json)
html = html.replace('__STATS__', stats_json)

with open('minsk_bank_dashboard_final.html', 'w', encoding='utf-8') as f:
    f.write(html)

import os
size_mb = os.path.getsize('minsk_bank_dashboard_final.html') / (1024 * 1024)
print(f"✅ Дашборд сохранён: minsk_bank_dashboard_final.html")
print(f"   Размер: {size_mb:.1f} МБ")
print(f"\n📂 Откройте в браузере!")

✅ Дашборд сохранён: minsk_bank_dashboard_final.html
   Размер: 12.8 МБ

📂 Откройте в браузере!


In [ ]:
# Загружаем данные
import json
with open('dashboard_data.json', 'r', encoding='utf-8') as f:
    dashboard_data = json.load(f)

houses_json = json.dumps(dashboard_data['houses'], ensure_ascii=False)
banks_json = json.dumps(dashboard_data['banks'], ensure_ascii=False)
bank_list_json = json.dumps(dashboard_data['bankList'], ensure_ascii=False)
stats_json = json.dumps(dashboard_data['stats'], ensure_ascii=False)

html_template = '''<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Банковская доступность — Минск</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
    * { margin: 0; padding: 0; box-sizing: border-box; }
    body { font-family: 'Segoe UI', Arial, sans-serif; background: #1a1a2e; color: #eee; }
    
    #container { display: flex; height: 100vh; }
    
    #sidebar {
        width: 370px; min-width: 370px;
        background: #16213e; padding: 20px;
        display: flex; flex-direction: column; gap: 15px;
        overflow-y: auto; border-right: 2px solid #0f3460;
    }
    
    #map { flex: 1; height: 100vh; }
    
    .panel {
        background: #0f3460; border-radius: 10px; padding: 15px;
    }
    
    .panel h3 {
        margin-bottom: 10px; color: #e94560; font-size: 16px;
        border-bottom: 1px solid #533483; padding-bottom: 5px;
    }
    
    button {
        width: 100%; padding: 8px; margin: 5px 0;
        background: #e94560; color: #eee; border: none;
        border-radius: 5px; cursor: pointer; font-size: 14px;
        font-weight: bold; transition: background 0.3s;
    }
    
    button:hover { background: #c23152; }
    
    .stats-row {
        display: flex; justify-content: space-between;
        font-size: 14px; margin: 3px 0;
    }
    
    .stats-dot {
        display: inline-block; width: 10px; height: 10px;
        border-radius: 50%; margin-right: 5px;
    }
    
    .legend-item {
        display: flex; align-items: center; margin: 5px 0; font-size: 13px;
    }
    
    .legend-color {
        width: 20px; height: 20px; border-radius: 4px; margin-right: 8px;
    }
    
    .switch-container {
        display: flex; gap: 5px; flex-wrap: wrap;
    }
    
    .switch-btn {
        flex: 1; padding: 8px; text-align: center;
        background: #1a1a2e; border: 1px solid #533483;
        border-radius: 5px; cursor: pointer; font-size: 12px;
        transition: all 0.3s;
    }
    
    .switch-btn.active {
        background: #e94560; border-color: #e94560;
    }
    
    .summary-box {
        text-align: center; padding: 10px;
        background: #1a1a2e; border-radius: 5px;
    }
    
    .summary-box .big-number {
        font-size: 36px; font-weight: bold; color: #e94560;
    }
    
    .checkbox-list {
        max-height: 180px; overflow-y: auto;
        background: #1a1a2e; border-radius: 5px; padding: 8px;
    }
    
    .checkbox-item {
        display: flex; align-items: center; padding: 4px 0;
        font-size: 13px; cursor: pointer;
    }
    
    .checkbox-item input {
        margin-right: 8px; cursor: pointer;
        accent-color: #e94560;
    }
    
    .bank-count {
        margin-left: auto; color: #999; font-size: 11px;
    }
    
    .select-all-row {
        display: flex; gap: 10px; margin-bottom: 5px;
    }
    
    .select-all-row button {
        flex: 1; padding: 4px; font-size: 11px; background: #533483;
    }
</style>
</head>
<body>

<div id="container">
    <div id="sidebar">
        <h2>Банковская доступность</h2>
        <p style="font-size:12px; color:#999;">Минск, 1661 точка обслуживания</p>
        
        <div class="panel">
            <h3>Режим отображения</h3>
            <div class="switch-container" id="modeSwitch">
                <div class="switch-btn active" data-mode="all">Все точки</div>
                <div class="switch-btn" data-mode="office">Отделения</div>
                <div class="switch-btn" data-mode="atm">Банкоматы</div>
            </div>
        </div>
        
        <div class="panel">
            <h3>Фильтр по банкам</h3>
            <div class="select-all-row">
                <button onclick="selectAllBanks()">Выбрать все</button>
                <button onclick="deselectAllBanks()">Снять все</button>
            </div>
            <div class="checkbox-list" id="bankCheckboxes"></div>
            <button onclick="applyBankFilter()">Применить фильтр</button>
        </div>
        
        <div class="panel">
            <h3>Статистика</h3>
            <div id="statsContent"></div>
        </div>
        
        <div class="panel">
            <h3>Легенда</h3>
            <div class="legend-item">
                <div class="legend-color" style="background:#2ecc71;"></div>
                Отлично (до 500м)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#f1c40f;"></div>
                Хорошо (500м–1км)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#e67e22;"></div>
                Удовлетворительно (1–2км)
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:#e74c3c;"></div>
                Плохо (более 2км)
            </div>
            <hr style="border-color:#533483; margin:10px 0;">
            <div class="legend-item">
                <div class="legend-color" style="background:darkgreen;"></div>
                Отделение
            </div>
            <div class="legend-item">
                <div class="legend-color" style="background:darkblue;"></div>
                Банкомат
            </div>
        </div>
    </div>
    
    <div id="map"></div>
</div>

<script>
var housesData = __HOUSES__;
var banksData = __BANKS__;
var bankList = __BANKLIST__;
var statsData = __STATS__;

var currentMode = 'all';
var filteredBanks = ['all'];

// Количество точек для каждого банка
var bankCounts = {};
for (var i = 0; i < banksData.length; i++) {
    var bn = banksData[i].bank_name;
    bankCounts[bn] = (bankCounts[bn] || 0) + 1;
}

// Сортируем банки по количеству точек
bankList.sort(function(a, b) {
    return (bankCounts[b] || 0) - (bankCounts[a] || 0);
});

// Создаём чекбоксы
var checkboxContainer = document.getElementById('bankCheckboxes');
for (var i = 0; i < bankList.length; i++) {
    var bank = bankList[i];
    var count = bankCounts[bank] || 0;
    
    var label = document.createElement('label');
    label.className = 'checkbox-item';
    
    var checkbox = document.createElement('input');
    checkbox.type = 'checkbox';
    checkbox.value = bank;
    checkbox.checked = true;
    checkbox.className = 'bank-checkbox';
    
    label.appendChild(checkbox);
    label.appendChild(document.createTextNode(bank));
    
    var span = document.createElement('span');
    span.className = 'bank-count';
    span.textContent = count;
    label.appendChild(span);
    
    checkboxContainer.appendChild(label);
}

function selectAllBanks() {
    var checks = document.querySelectorAll('.bank-checkbox');
    for (var i = 0; i < checks.length; i++) checks[i].checked = true;
}

function deselectAllBanks() {
    var checks = document.querySelectorAll('.bank-checkbox');
    for (var i = 0; i < checks.length; i++) checks[i].checked = false;
}

// Карта
var map = L.map('map', {
    center: [53.9, 27.56],
    zoom: 12,
    zoomControl: true,
});

L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', {
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>',
    maxZoom: 18,
}).addTo(map);

var housesLayer = L.featureGroup().addTo(map);
var banksLayer = L.featureGroup().addTo(map);

var categoryColors = {
    'Отлично (до 500м)': '#2ecc71',
    'Хорошо (500м-1км)': '#f1c40f',
    'Удовлетворительно (1-2км)': '#e67e22',
    'Плохо (более 2км)': '#e74c3c',
    'Нет данных': '#95a5a6',
};

// Показываем каждый N-й дом для производительности
var HOUSE_SAMPLE = 5;

function getDistField() { return 'dist_' + currentMode; }
function getCatField() { return 'cat_' + currentMode; }

function renderHouses() {
    housesLayer.clearLayers();
    
    var distField = getDistField();
    var catField = getCatField();
    
    for (var i = 0; i < housesData.length; i += HOUSE_SAMPLE) {
        var house = housesData[i];
        var dist = house[distField];
        var cat = house[catField];
        
        if (dist === null || dist === undefined) continue;
        
        var color = categoryColors[cat] || '#95a5a6';
        var marker = L.circleMarker([house.lat, house.lon], {
            radius: 4,
            fillColor: color,
            color: color,
            weight: 0.5,
            fillOpacity: 0.6,
        });
        
        marker.bindPopup(
            '<b>' + cat + '</b><br>' +
            'Расстояние: ' + Math.round(dist) + 'м<br>' +
            '<small>Ближ. офис: ' + (house.nearest_office || '—') + '</small><br>' +
            '<small>Ближ. банкомат: ' + (house.nearest_atm || '—') + '</small>'
        );
        
        housesLayer.addLayer(marker);
    }
}

function renderBanks() {
    banksLayer.clearLayers();
    
    for (var i = 0; i < banksData.length; i++) {
        var bank = banksData[i];
        
        if (filteredBanks.indexOf('all') === -1 && 
            filteredBanks.indexOf(bank.bank_name) === -1) continue;
        
        if (currentMode === 'office' && bank.type !== 'office') continue;
        if (currentMode === 'atm' && bank.type !== 'atm') continue;
        
        var color = bank.type === 'office' ? 'darkgreen' : 'darkblue';
        var marker = L.circleMarker([bank.lat, bank.lon], {
            radius: 6,
            fillColor: color,
            color: '#fff',
            weight: 1.5,
            fillOpacity: 0.95,
        });
        
        marker.bindPopup(
            '<b>' + bank.bank_name + '</b><br>' +
            (bank.type === 'office' ? 'Отделение' : 'Банкомат') + '<br>' +
            (bank.address || '')
        );
        
        banksLayer.addLayer(marker);
    }
}

function renderAll() {
    renderHouses();
    renderBanks();
    updateStats();
}

function updateStats() {
    var modeStats = statsData[currentMode];
    var total = modeStats.total;
    
    document.getElementById('statsContent').innerHTML =
        '<div class="summary-box">' +
        '<div class="big-number">' + total.toLocaleString() + '</div>' +
        '<div>домов проанализировано</div>' +
        '</div>' +
        '<div style="margin-top:10px;">' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#2ecc71;"></span>Отлично</span>' +
        '<span>' + modeStats.excellent.toLocaleString() + ' (' + (modeStats.excellent/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#f1c40f;"></span>Хорошо</span>' +
        '<span>' + modeStats.good.toLocaleString() + ' (' + (modeStats.good/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#e67e22;"></span>Удовл.</span>' +
        '<span>' + modeStats.ok.toLocaleString() + ' (' + (modeStats.ok/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '<div class="stats-row">' +
        '<span><span class="stats-dot" style="background:#e74c3c;"></span>Плохо</span>' +
        '<span>' + modeStats.bad.toLocaleString() + ' (' + (modeStats.bad/total*100).toFixed(1) + '%)</span>' +
        '</div>' +
        '</div>';
}

// Переключатели режима
document.querySelectorAll('#modeSwitch .switch-btn').forEach(function(btn) {
    btn.addEventListener('click', function() {
        document.querySelectorAll('#modeSwitch .switch-btn').forEach(function(b) { b.classList.remove('active'); });
        this.classList.add('active');
        currentMode = this.dataset.mode;
        renderAll();
    });
});

// Фильтр банков
function applyBankFilter() {
    var checks = document.querySelectorAll('.bank-checkbox');
    var selected = [];
    
    for (var i = 0; i < checks.length; i++) {
        if (checks[i].checked) {
            selected.push(checks[i].value);
        }
    }
    
    // Если выбраны все — показываем все банки
    if (selected.length === bankList.length) {
        filteredBanks = ['all'];
    } else {
        filteredBanks = selected;  // может быть пустым — тогда ничего не покажем
    }
    renderAll();
}

renderAll();
</script>
</body>
</html>'''

# Подставляем данные
html = html_template.replace('__HOUSES__', houses_json)
html = html.replace('__BANKS__', banks_json)
html = html.replace('__BANKLIST__', bank_list_json)
html = html.replace('__STATS__', stats_json)

with open('minsk_bank_dashboard_v3.html', 'w', encoding='utf-8') as f:
    f.write(html)

import os
size_mb = os.path.getsize('minsk_bank_dashboard_v3.html') / (1024 * 1024)
print(f"✅ Дашборд v3 сохранён: minsk_bank_dashboard_v3.html")
print(f"   Размер: {size_mb:.1f} МБ")
print(f"   Выборка домов: каждый 5-й (~{len(dashboard_data['houses']) // 5} точек)")
print(f"\n📂 Откройте в браузере!")

✅ Дашборд v3 сохранён: minsk_bank_dashboard_v3.html
   Размер: 12.8 МБ
   Выборка домов: каждый 5-й (~7879 точек)

📂 Откройте в браузере!
